# 04 · Decoding-strategy sweep (temperature / top-k / top-p / repetition penalty)
Runs **your** grid of decoding settings on the held-out validation prompts (never the official test set): the same
fixed prompt subset for every setting, sampled settings averaged over several seeds (mean ± std). It logs BERTScore /
ROUGE-L / BLEU-4, plus diagnostics for the analysis: `repeat_4gram` (degenerate loops), `distinct_2` (diversity),
`avg_words` / `frac_short` (did it reach 128 words?). Results are appended to `RUNS_DIR/<run>/decode_sweep.jsonl`,
and re-running skips settings that are already done.

Course rule §17.5: choosing the grid and drawing conclusions is yours; this notebook only runs and records it.

In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
!pip -q install tokenizers bert-score rouge-score sacrebleu
RUN = 'A_qwen_dense_full'     # which Phase-1 run to decode with
WHICH = 'best'                # best | final | step_XXXXXX
if RUN.startswith('C_gdn_hybrid'):
    !pip -q install flash-linear-attention

In [ ]:
# copy tokenised data from Drive to fast local disk (once per runtime); the training file is TRAIN_BIN if set
import shutil, os
os.makedirs(LOCAL_DATA, exist_ok=True)
_train_files = [globals()['TRAIN_BIN']] if 'TRAIN_BIN' in globals() else []
for f in _train_files + ['val.bin', 'tokenizer.json', 'val_prompts.json', 'val_refs.json', 'val_srcs.json', 'test.json']:
    src, dst = f'{DRIVE_DATA}/{f}', f'{LOCAL_DATA}/{f}'
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src)):
        shutil.copyfile(src, dst)
!ls -lh {LOCAL_DATA}

In [ ]:
import json
from lm_lib.trainer import load_trained
from lm_lib.decode_sweep import run_sweep, results_table
model, tok, cfg, mtype = load_trained(f'{RUNS_DIR}/{RUN}', WHICH)
prompts = {k: v['text'] for k, v in json.load(open(f'{LOCAL_DATA}/val_prompts.json')).items()}
refs = json.load(open(f'{LOCAL_DATA}/val_refs.json'))
OUT = f'{RUNS_DIR}/{RUN}/decode_sweep_{WHICH}.jsonl'

In [ ]:
# ---- define YOUR grid: one dict per setting (unspecified keys default to temperature=0, top_k=0, top_p=1, rep=1) ----
GRID = [
    dict(temperature=0.0),                                  # greedy baseline
    # dict(temperature=..., top_k=..., top_p=..., repetition_penalty=...),
]
results = run_sweep(model, tok, mtype, cfg, prompts, refs, GRID, OUT,
                    n_prompts=200,        # fixed subset of the 300 val prompts
                    seeds=(0, 1, 2),      # sampled settings: mean +- std over these seeds
                    gen=dict(n_words=128, max_new_tokens=256, batch_size=64))

In [ ]:
df = results_table(OUT)
df

In [ ]:
# plots (ancillary): BERTScore vs temperature for each top_p, and the repetition diagnostic
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for (tp, rp), g in df.groupby(['top_p', 'repetition_penalty']):
    g = g.sort_values('temperature')
    ax[0].errorbar(g.temperature, g.bertscore_f1, yerr=g.bertscore_f1_std, marker='o', capsize=3, label=f'top_p={tp}, rep={rp}')
    ax[1].plot(g.temperature, g.repeat_4gram, marker='o', label=f'top_p={tp}, rep={rp}')
ax[0].set_xlabel('temperature'); ax[0].set_ylabel('BERTScore F1 (val)'); ax[0].legend(fontsize=7)
ax[1].set_xlabel('temperature'); ax[1].set_ylabel('repeated 4-grams'); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# use your chosen setting for the test set (same code path as notebook 02)
CHOSEN = dict(temperature=0.0, top_k=0, top_p=1.0, repetition_penalty=1.0)   # <- your pick from the table above
import time
from lm_lib.generate import generate_continuations
from lm_lib.submit import write_submission
test = json.load(open(f'{LOCAL_DATA}/test.json'))
test_preds = generate_continuations(model, tok, mtype, {k: v['text'] for k, v in test.items()}, cfg.seq_len, cfg.src_len,
                                    sampling=CHOSEN, n_words=128, max_new_tokens=256, batch_size=64)
tag = f"{RUN}_{WHICH}_decode_{time.strftime('%m%d_%H%M')}"
zip_path = write_submission(test_preds, test, f'{RUNS_DIR}/{RUN}/submissions/{tag}')
json.dump(dict(sampling=CHOSEN, ckpt=WHICH), open(f'{RUNS_DIR}/{RUN}/submissions/{tag}/settings.json', 'w'))